# Ejecutar Pipeline — modelo-riesgo

Este notebook registra y lanza el pipeline de SageMaker generado por el agente `ml-pipeline-builder`.

**Kernel:** Python 3 (ipykernel)  
**Ambiente:** SageMaker Studio Space `nave-demo-space`

---

## Antes de empezar

La imagen de Studio trae el **SDK v3** (paquetes `sagemaker-core`, `-train`, `-serve`, `-mlops`).
El pipeline de este repo está escrito con la **API v2** (`sagemaker.workflow`, `Estimator`), y
v3 **no tiene capa de compatibilidad**: la clase `Estimator` ya no existe.

Por eso la celda 1 desinstala v3 y ancla v2. Se corre **una sola vez por espacio**,
y **hay que reiniciar el kernel después**.

## 1. Instalación (una sola vez → luego REINICIAR KERNEL)

In [ ]:
# La imagen de Studio trae el SDK v3, que es INCOMPATIBLE con este pipeline (API v2).
# Hay que sacar los cinco paquetes del namespace, no solo 'sagemaker':
# si queda alguno, el directorio sagemaker/ sigue siendo namespace package
# y la instalacion v2 no se asienta limpia.

%pip uninstall -y sagemaker sagemaker-core sagemaker-train sagemaker-serve sagemaker-mlops
%pip install --quiet "sagemaker==2.*"

print("=" * 60)
print("AHORA REINICIA EL KERNEL:  menu Kernel -> Restart Kernel")
print("Re-ejecutar esta celda NO basta. Despues sigue desde la celda 2.")
print("=" * 60)

## 2. Verificación (después del reinicio)

Si `__version__` falla con `AttributeError`, quedaron restos de v3: repetí la celda 1 y reiniciá otra vez.

In [ ]:
import sys
import sagemaker

print(f"Python  : {sys.executable}")
print(f"SDK path: {sagemaker.__file__}")
print(f"SDK ver : {sagemaker.__version__}")   # debe ser 2.x

assert sagemaker.__version__.startswith("2."), (
    f"Se esperaba SDK v2, hay {sagemaker.__version__}. "
    "Repetir celda 1 y reiniciar el kernel."
)

from sagemaker.workflow.pipeline import Pipeline
print("sagemaker.workflow OK")

## 3. Cargar el pipeline del repo

In [ ]:
import os
import sys

REPO_ROOT = os.path.expanduser("~/ar3306-nave-mvp-aceleradores")
PIPELINE_PATH = f"{REPO_ROOT}/ml/pipelines/modelo-riesgo/pipeline.py"

# El repo va al path porque pipeline.py suele importar modulos hermanos del proyecto
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

# Fallar aca con un mensaje claro es mejor que un ImportError opaco mas adelante
assert os.path.isfile(PIPELINE_PATH), f"No existe: {PIPELINE_PATH}"

print(f"Repo root    : {REPO_ROOT}")
print(f"Pipeline file: {PIPELINE_PATH}")

In [ ]:
# IMPORTANTE: cambia el cwd al directorio de los scripts ANTES de crear el pipeline.
import importlib.util

SRC_DIR = REPO_ROOT + "/ml/src/modelo-riesgo"
os.chdir(SRC_DIR)
print("cwd:", os.getcwd())

spec = importlib.util.spec_from_file_location("pipeline_modelo_riesgo", PIPELINE_PATH)
pipeline_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(pipeline_module)

pipeline = pipeline_module.create_pipeline()
print("Pipeline creado:", pipeline.name)

## 4. Registrar y ejecutar

In [ ]:
# upsert = crear si no existe, actualizar si ya existe
# Los tags son obligatorios por el SCP de la cuenta sandbox de Nubiral
ROLE = "arn:aws:iam::015319782619:role/ar3306-nave-aceleradores-sagemaker-execution-role"

TAGS = [
    {"Key": "owner",     "Value": "santiago.castro@nubiral.com"},
    {"Key": "project",   "Value": "ar3306-nave-aceleradores"},
    {"Key": "createdBy", "Value": "santiago.castro@nubiral.com"},
    {"Key": "team",      "Value": "pod7"},
    {"Key": "deadline",  "Value": "2026-12-31"},
]

upsert_response = pipeline.upsert(role_arn=ROLE, tags=TAGS)
print(f"Pipeline registrado: {upsert_response['PipelineArn']}")

In [ ]:
execution = pipeline.start()

print(f"Ejecucion iniciada: {execution.arn}")
print()
print("Monitorea el progreso en:")
print("SageMaker Studio -> Pipelines -> modelo-riesgo-pipeline")

## 5. (Opcional) Esperar y revisar el estado

`wait()` levanta excepción si el pipeline falla o si se agota el tiempo. En ambos casos
conviene igual ver el detalle de los steps, así que va en `try/finally`.

In [ ]:
print("Esperando que complete (puede tardar 10-15 min)...")

try:
    # delay=30s x max_attempts=60 -> hasta 30 min de espera
    execution.wait(delay=30, max_attempts=60)
    print("Ejecucion terminada.")
except Exception as e:
    print(f"wait() termino con: {type(e).__name__}: {e}")
finally:
    print("\nResumen de steps:")
    for step in execution.list_steps():
        print(f"  {step['StepName']:30s} {step['StepStatus']}")
        if step.get("FailureReason"):
            print(f"    -> {step['FailureReason']}")